# HW2 — Profile & Optimize an Autoregressive Decode Loop

**Lecture mapping:** L1 §07 (Profiling) · L2 §01 (prefill/decode, KV cache) · L2 §03 (engine optimizations)

The harness below defines a deliberately slow greedy-decode loop (**V0**): no KV
cache (it recomputes the whole sequence every step), fp32 eager, and a host sync
every step. Profile it, find the bottlenecks, and write a fast, numerically
identical replacement.

## What you implement

| Part | Function |
|------|----------|
| 1 | `profile` — wrap a loop in `torch.profiler`, print a table, export a Chrome trace |
| 2 | `optimized_loop` — fast greedy decode, same tokens as V0 (fp32) |
| 3 | `generate_optimized` — build, warm up, and time your optimized generation |
| 4 | Writeup Q1–Q4 |

## Speedup targets

`optimized_loop` end-to-end vs the V0 baseline:

| Speedup vs V0 | |
|---------------|--|
| ≥ 4.0× | excellent |
| ≥ 3.0× | good |
| ≥ 1.5× | some speedup |
| < 1.5× | little to no speedup |

The speedup only counts if `optimized_loop` passes the fp32 correctness check and
the timed run is on a GPU.

## Rules

- PyTorch + `requirements.txt` only — no vLLM / TensorRT-LLM / SGLang.
- Don't edit the harness cell.
- `optimized_loop` must reproduce the baseline's greedy tokens exactly on the same
  fp32 model. `generate_optimized` may switch to bf16 for the timed run.

## Where to look

- **KV cache** (L2 §01) — the baseline throws it away every step. Biggest win.
- **Host syncs** (L1 §07) — `.item()` on the critical path serializes CPU↔GPU.
- **`torch.compile` / CUDA graphs** (L2 §03) — fuse ops, cut launch overhead.
- **dtype** — bf16 for the timed run (small on a model this size).

## Cell types

- **DO NOT EDIT** — fixed harness.
- **YOUR IMPLEMENTATION** — replace `raise NotImplementedError`.
- **SELF-CHECK** — asserts that must pass.
- **WRITEUP** — answer in the markdown cell.

Run top-to-bottom on the GPU. Submit the executed notebook plus the files under
`results/`.

In [ ]:
# Colab setup (run first). Pin transformers to the 4.x KV-cache (Cache) API the
# harness targets; Colab preinstalls torch / numpy / matplotlib already.
!pip install -q "transformers>=4.42,<5"

## The fixed harness (DO NOT EDIT)

Tiny 2-layer Llama, the V0 baseline, the correctness check, the timing helper,
and the speedup targets.

In [ ]:
# DO NOT EDIT — editing this cell invalidates your speedup numbers.
import os, time

import torch
from transformers import LlamaConfig, LlamaForCausalLM

RESULTS_DIR = os.path.join("results", "hw2")
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 0
PROMPT_LEN = 64
MAX_NEW_TOKENS = 128

# Speedup targets (optimized end-to-end vs V0 baseline).
TIERS = [
    (4.0, ">= 4.0x  (excellent)"),
    (3.0, ">= 3.0x  (good)"),
    (1.5, ">= 1.5x  (some speedup)"),
    (0.0, "< 1.5x  (little to no speedup)"),
]


def device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def _config() -> LlamaConfig:
    # Small enough to iterate fast, real enough to profile meaningfully.
    return LlamaConfig(
        vocab_size=32000,
        hidden_size=512,
        intermediate_size=1376,
        num_hidden_layers=2,
        num_attention_heads=8,
        num_key_value_heads=8,
        max_position_embeddings=4096,
    )


def build_model_and_input(dtype: torch.dtype = torch.float32):
    """Return (model, input_ids) on the active device with fixed random weights.

    The same SEED is used every call, so the V0 baseline and your optimized run
    see identical weights and the same prompt.
    """
    torch.manual_seed(SEED)
    model = LlamaForCausalLM(_config()).to(device=device(), dtype=dtype).eval()
    input_ids = torch.randint(0, 32000, (1, PROMPT_LEN), device=device())
    return model, input_ids


@torch.no_grad()
def baseline_loop(model, input_ids, max_new_tokens: int) -> torch.Tensor:
    """V0 — intentionally slow. Greedy decode with NO KV cache: every step
    re-runs the model over the whole growing sequence (O(n^2)), and forces a
    host sync each step (the `.item()`) — exactly the anti-patterns from lecture.

    Returns the generated token ids, shape (1, max_new_tokens).
    """
    generated = input_ids
    out_tokens = []
    for _ in range(max_new_tokens):
        out = model(input_ids=generated, use_cache=False)
        next_tok = out.logits[:, -1:, :].argmax(dim=-1)
        _ = next_tok.item()                       # forced host sync every step
        generated = torch.cat([generated, next_tok], dim=1)
        out_tokens.append(next_tok)
    return torch.cat(out_tokens, dim=1)


def time_loop(loop_fn, *args, n_warmup: int = 1, n_iters: int = 3) -> float:
    """Average seconds per full generation of `loop_fn(*args)`."""
    for _ in range(n_warmup):
        loop_fn(*args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(n_iters):
            loop_fn(*args)
        end.record()
        torch.cuda.synchronize()
        return (start.elapsed_time(end) / 1e3) / n_iters
    t0 = time.perf_counter()
    for _ in range(n_iters):
        loop_fn(*args)
    return (time.perf_counter() - t0) / n_iters


def check_correctness(reference: torch.Tensor, candidate: torch.Tensor) -> bool:
    """Greedy decoding is deterministic, so a correct optimized loop must
    reproduce the baseline's token ids EXACTLY when given the same fp32 model."""
    return (reference.shape == candidate.shape
            and torch.equal(reference.cpu(), candidate.cpu()))


def speedup_tier(speedup: float) -> str:
    for threshold, label in TIERS:
        if speedup >= threshold:
            return label
    return TIERS[-1][1]


print(f"device={device()}  prompt_len={PROMPT_LEN}  new_tokens={MAX_NEW_TOKENS}")

## Part 1 — profile a generation loop

See the L1 §07 `torch.profiler` snippet. You will use this on both the baseline
and your optimized loop, and read the traces at
[ui.perfetto.dev](https://ui.perfetto.dev).

In [ ]:
# TODO-CELL: profile
# YOUR IMPLEMENTATION
def profile(loop_fn, model, input_ids, max_new_tokens: int, trace_path: str) -> None:
    """Run `loop_fn(model, input_ids, max_new_tokens)` under torch.profiler.

    Requirements:
      * Profile both CPU and CUDA activity (ProfilerActivity).
      * Print a short table sorted by self CUDA time (key_averages().table(...))
        - on CPU-only machines sort by self CPU time instead.
      * Export a Chrome trace to `trace_path`.
    """
    on_cuda = torch.cuda.is_available()

    activities = [torch.profiler.ProfilerActivity.CPU]
    if on_cuda:
        activities.append(torch.profiler.ProfilerActivity.CUDA)

    with torch.profiler.profile(activities=activities) as prof:
        loop_fn(model, input_ids, max_new_tokens)
        if on_cuda:
            torch.cuda.synchronize()   # let the queued kernels finish before we stop recording

    sort_key = "self_cuda_time_total" if on_cuda else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=15))
    prof.export_chrome_trace(trace_path)

## Part 2 — a fast, correct greedy decode loop

Return the same token ids as `baseline_loop` for the same fp32 model — shape
`(1, max_new_tokens)`. Start with what the baseline recomputes every step and
what it forces onto the host.

In [ ]:
# TODO-CELL: optimized_loop
# YOUR IMPLEMENTATION
@torch.no_grad()
def optimized_loop(model, input_ids, max_new_tokens: int) -> torch.Tensor:
    """Greedy decode, but fast. Same tokens as baseline_loop, much less work.

    Two changes vs V0:
      * KV cache: prefill the prompt once, then feed only the SINGLE new token
        each step (process 1 position, not the whole growing sequence -> O(n)).
      * No host sync: keep argmax on-device, collect tokens in a list, and
        concat at the very end. No `.item()` on the hot path.
    """
    # Prefill: run the whole prompt once, keep the cache.
    out = model(input_ids=input_ids, use_cache=True)
    past = out.past_key_values
    next_tok = out.logits[:, -1:, :].argmax(dim=-1)   # first generated token
    out_tokens = [next_tok]

    # Decode: feed only the last token + the cache; the model attends to the
    # cached prompt+history, so each step processes exactly one position.
    for _ in range(max_new_tokens - 1):
        out = model(input_ids=next_tok, past_key_values=past, use_cache=True)
        past = out.past_key_values
        next_tok = out.logits[:, -1:, :].argmax(dim=-1)
        out_tokens.append(next_tok)

    return torch.cat(out_tokens, dim=1)   # (1, max_new_tokens)

## Part 3 — build + time the optimized generation

Build the model however you like (dtype, `torch.compile`, CUDA graphs), run
`optimized_loop`, and return `(elapsed_seconds, generated_token_ids)`.

`elapsed_seconds` must be a warmed-up, timed measurement (use `time_loop`); it's
what gets compared to V0. Numerics-changing tricks (e.g. bf16) are fine here —
correctness is checked separately on the fp32 path.

In [ ]:
# TODO-CELL: generate_optimized
# YOUR IMPLEMENTATION
def generate_optimized(max_new_tokens: int = MAX_NEW_TOKENS):
    """Return (elapsed_seconds, generated_token_ids) for your fastest setup.

    Fastest reliable path on this tiny model:
      * bf16 for the timed run (half the bytes per weight/activation -> less HBM
        traffic on a bandwidth-bound decode; numerics are checked separately on
        the fp32 path, so this is allowed here).
      * the KV-cache loop from Part 2.

    `torch.compile` is left out on purpose: the HF decode loop changes shape
    between prefill (seq=PROMPT_LEN) and decode (seq=1) and grows its cache every
    step, so compiling it tends to recompile / graph-break and adds compile
    latency without a clear win on a 2-layer model. The big lever here is the KV
    cache (O(n^2) -> O(n)); bf16 is the cheap bonus.

    Caveat (see Q3): the optimized loop is itself launch-overhead-bound, so the
    >=4x tier is a target, not a guarantee. If a GPU run lands in [3x, 4x), the
    measured fallback is to collapse the per-step launch storm with
    torch.compile(model, dynamic=True) or a hand-rolled CUDA graph over the
    steady-state decode step (HW3) - measure it one-at-a-time and record the
    number in the Q2 table rather than assuming it helps on this tiny model.
    """
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model, input_ids = build_model_and_input(dtype)

    # n_warmup=2 so cuBLAS autotune / first-call costs are excluded from the
    # steady-state timing.
    elapsed = time_loop(optimized_loop, model, input_ids, max_new_tokens, n_warmup=2)
    tokens = optimized_loop(model, input_ids, max_new_tokens)
    return elapsed, tokens

## Self-check (small + fast, runs anywhere)

In [ ]:
# SELF-CHECK — DO NOT EDIT.
_n = 24  # short, for speed
_model, _ids = build_model_and_input(torch.float32)

_ref = baseline_loop(_model, _ids, _n)
_cand = optimized_loop(_model, _ids, _n)
assert tuple(_cand.shape) == (1, _n), f"expected shape (1, {_n}), got {tuple(_cand.shape)}"
assert check_correctness(_ref, _cand), \
    "optimized_loop must reproduce the baseline greedy tokens EXACTLY (fp32)"
print("optimized_loop matches baseline   PASS")

_trace = os.path.join(RESULTS_DIR, "trace_check.json")
if os.path.exists(_trace):
    os.remove(_trace)
profile(baseline_loop, _model, _ids, 4, _trace)
assert os.path.exists(_trace) and os.path.getsize(_trace) > 0, \
    "profile() must write a non-empty Chrome trace file"
print("profile() writes a Chrome trace   PASS")
print()
print("All checks passed")

## The full run — baseline vs optimized (GPU)

Prints your speedup and writes the two traces for the writeup.

In [ ]:
# DO NOT EDIT — the full run.
N_NEW = MAX_NEW_TOKENS if torch.cuda.is_available() else 32
if N_NEW != MAX_NEW_TOKENS:
    print("CPU detected — trimmed to 32 new tokens as a smoke test. "
          "REPORTED NUMBERS MUST COME FROM A GPU RUN.")

# Baseline (V0), fp32 — also the correctness reference.
model_fp32, input_ids = build_model_and_input(torch.float32)
base_t = time_loop(baseline_loop, model_fp32, input_ids, N_NEW)
ref = baseline_loop(model_fp32, input_ids, N_NEW)
print(f"baseline V0: {base_t*1e3:.1f} ms/generation")

# Correctness of your loop on the SAME fp32 model.
cand = optimized_loop(model_fp32, input_ids, N_NEW)
ok = check_correctness(ref, cand)
print(f"optimized_loop correctness (fp32): {'PASS' if ok else 'FAIL'}")

# Optimized end-to-end timing (your choice of dtype/compile/graphs).
opt_t, _ = generate_optimized(N_NEW)
speedup = base_t / opt_t
print(f"optimized:   {opt_t*1e3:.1f} ms/generation")
print(f"speedup: {speedup:.2f}x  ->  {speedup_tier(speedup)}")
if not ok:
    print("WARNING: correctness FAILED — the speedup does not count.")

# Two traces for the writeup: baseline vs optimized.
profile(baseline_loop, model_fp32, input_ids, 32,
        os.path.join(RESULTS_DIR, "trace_baseline.json"))
profile(optimized_loop, model_fp32, input_ids, 32,
        os.path.join(RESULTS_DIR, "trace_optimized.json"))
print("wrote traces to results/hw2/ — open them at https://ui.perfetto.dev")

---
## WRITEUP (be concrete and quantitative)

### Q1

From your **baseline trace**, what dominates the time? Name the specific symptom
you saw in the profiler (e.g. kernel-launch gaps, a long run of tiny kernels,
host syncs) and explain it.

**Your answer:**

> Two symptoms dominate, both designed into V0:
>
> 1. **Redundant full-sequence recompute (no KV cache).** Each step calls the
>    model over the *entire* growing sequence, so the self-CUDA-time table is
>    topped by the linear-layer and attention matmuls (`*mm*` / `*gemm*` /
>    scaled-dot-product-attention kernels), and their cost grows step over step.
>    Total work is O(n^2): for PROMPT_LEN=64 and 128 new tokens the baseline
>    processes ~16,000 token-positions vs ~190 for the KV-cache loop.
>    *(Top self-CUDA op in my trace: `[run: name]` at `[run: %]` of CUDA time.)*
> 2. **A host sync every step (`.item()`).** The trace shows repeated
>    `cudaStreamSynchronize` / `cudaDeviceSynchronize` calls and visible
>    **launch gaps** on the GPU timeline: the CPU blocks waiting for the GPU to
>    return one integer before it can queue the next step, so the GPU sits idle
>    between bursts instead of staying fed.
>
> The net picture is a bandwidth/compute-heavy forward repeated far too often,
> with the CPU and GPU ping-ponging once per token instead of overlapping.

### Q2

List each optimization you applied and the speedup it contributed - measure them
**one at a time** (baseline -> +KV cache -> +compile -> ...). A small table is ideal.

**Your answer:**

> Measured one at a time, each row built on the row above (same prompt, same
> MAX_NEW_TOKENS, warmed up with `time_loop`). To reproduce a row, swap the dtype
> / loop in a scratch cell and call `time_loop(...)`; the baseline number is the
> V0 print from the full-run cell.
>
> | Variant | ms / generation | Speedup vs V0 | Cumulative |
> |---|---|---|---|
> | V0 baseline (fp32, no cache, `.item()` each step) | `[run]` | 1.00x | 1.00x |
> | + KV cache (fp32, no host sync) | `[run]` | `[run]` | `[run]` |
> | + bf16 (KV cache, fp32 -> bf16) | `[run]` | `[run]` | `[run]` |
>
> Notes:
> - The KV cache also removes the per-step `.item()` host sync (the V0 loop
>   syncs; the optimized loop keeps everything on-device until the final concat),
>   so that row folds two wins together: O(n^2)->O(n) compute **and** no sync.
> - `torch.compile` is not in the shipped `generate_optimized` (it recompiles
>   across the prefill/decode shape change and the growing cache on this 2-layer
>   model; compile latency without a clear steady-state win). If you try it, add
>   a `+torch.compile` row here with the measured number.
> - Final shipped speedup (bf16 + KV cache): `[run]x` -> tier `[run]`.

### Q3

Which single change had the biggest impact, and why does it help THIS workload
(128 short decode steps on a tiny model) specifically?

**Your answer:**

> The **KV cache** is the biggest single change. V0 re-runs the full model over
> the whole sequence every step, so per-step cost grows with the context and the
> total is O(n^2) in the number of generated tokens. With a KV cache, the prompt
> is encoded once (prefill) and every decode step processes exactly **one** new
> token, attending to cached keys/values - O(n) total. That cuts processed
> token-positions from ~16,000 to ~190 (an ~85x **work** reduction).
>
> Important: ~85x is the work cut, NOT the wall-time speedup. Both loops still
> issue exactly 128 `forward()` calls, and on a model this tiny each call is
> dominated by a fixed per-call floor (Python dispatch + dozens of kernel
> launches + the weight HBM read) that is paid by **both** loops and does not
> cancel from the ratio. So the optimized loop is itself launch-overhead-bound -
> the regime HW3 studies - and its wall time is a near-fixed floor. The realized
> speedup comes from what V0 pays *on top of* that floor and the optimized loop
> does not: the growing O(n^2) attention compute, and (crucially) the per-step
> `.item()` host sync that serializes CPU and GPU every step. That is why the
> measured speedup is far below 85x and the >=4x tier is a target the run may
> land just above or below. bf16 helps too (half the bytes per weight/activation
> on a bandwidth-bound step), but it shrinks only the matmul/bandwidth fraction,
> not the launch floor - a constant-factor bonus on top of the KV cache.

### Q4

Your decode loop is memory-bandwidth-bound (L1 §06, L2 §01). Which of your
optimizations attack memory traffic vs CPU/launch overhead? Which kind mattered
more here, and why?

**Your answer:**

> - **Memory-traffic attacks:** the KV cache (each step reads one token's worth
>   of activations + the cached K/V instead of re-reading and recomputing the
>   whole sequence) and bf16 (half the bytes per weight and activation moved
>   to/from HBM). A batch-1 decode step does almost no arithmetic per byte
>   loaded - it is far left on the HW1 roofline (low arithmetic intensity), so it
>   is limited by HBM bandwidth, and both of these cut bytes moved.
> - **CPU/launch-overhead attacks:** removing the per-step `.item()` host sync
>   (the CPU no longer blocks on the GPU each step, so launches overlap with
>   execution). `torch.compile` / CUDA graphs would attack launch overhead
>   further by fusing/replaying kernels - that is exactly what HW3 does.
>
> Which mattered more: the **memory-traffic** fix (KV cache), by a wide margin,
> because it also changes the asymptotics (O(n^2) -> O(n)). The sync removal is a
> real but bounded per-step win, and pure launch overhead is the dominant cost
> only once the redundant compute is already gone (the regime HW3 isolates). On
> V0, the recompute swamps everything, so attacking memory traffic wins here.